<a href="https://colab.research.google.com/github/jakubs692/University/blob/main/DNN/dnn02-MSLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# MSLE and gradients

Today we're implementing Gradient Descent by hand.
We'll also look at the Mean Squared Logarithmic Error (MSLE).

In [59]:
%matplotlib inline

from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from tqdm.auto import tqdm

NDArray = np.ndarray[Any, Any]

np.set_printoptions(precision=4, suppress=True)
np.random.seed(357)

## Loading and converting data

Same data as before, cleaned up. We'll use NumPy arrays instead of Pandas DataFrames to simplify the implementation.

In [60]:
!wget --no-verbose -O lab1_apartments_train_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv
!wget --no-verbose -O lab1_apartments_test_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv
!head lab1_apartments_train_clean.csv lab1_apartments_test_clean.csv

2026-10-15 21:12:59 URL:https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv [6187/6187] -> "lab1_apartments_train_clean.csv" [1]
2026-10-15 21:13:00 URL:https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv [6223/6223] -> "lab1_apartments_test_clean.csv" [1]
==> lab1_apartments_train_clean.csv <==
area,district,bedrooms,bathrooms,build_year,parking,price
104,mokotowo,2,2,1940,1,780094
43,ochotowo,1,1,1970,1,346912
128,grodziskowo,3,2,1916,1,523466
112,mokotowo,3,2,1920,1,830965
149,mokotowo,3,3,1977,0,1090479
80,ochotowo,2,2,1937,0,599060
58,ochotowo,2,1,1922,0,463639
23,ochotowo,1,1,1929,0,166785
40,mokotowo,1,1,1973,0,318849

==> lab1_apartments_test_clean.csv <==
area,district,bedrooms,bathrooms,build_year,parking,price
71,wolowo,2,2,1912,1,322227
45,mokotowo,1,1,1938,0,295878
38,mokotowo,1,1,1999,1,306530
70,ochotowo,2,2,1980,1,553641
136,mokotowo,3,2,1939,1,985348
128,wolowo,3,2,1983,1,695726
23,grodziskowo,1,1,1975,0,99751
117,mokotowo,3,2,1942,0,

In [61]:
def load_dataset(path: str) -> tuple[NDArray, NDArray]:
    """
    Returns (x, y) where:
    - x: input features, shape (n_apartments, n_features)
    - y: prices, shape (n_apartments,)
    """
    data = pd.read_csv(path)
    print(data.head())
    y = data["price"].to_numpy()
    x = data.loc[:, data.columns != "price"].to_numpy()
    return x, y

x_train, y_train = load_dataset("lab1_apartments_train_clean.csv")
x_test, y_test = load_dataset("lab1_apartments_test_clean.csv")

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

   area     district  bedrooms  bathrooms  build_year  parking    price
0   104     mokotowo         2          2        1940        1   780094
1    43     ochotowo         1          1        1970        1   346912
2   128  grodziskowo         3          2        1916        1   523466
3   112     mokotowo         3          2        1920        1   830965
4   149     mokotowo         3          3        1977        0  1090479
   area  district  bedrooms  bathrooms  build_year  parking   price
0    71    wolowo         2          2        1912        1  322227
1    45  mokotowo         1          1        1938        0  295878
2    38  mokotowo         1          1        1999        1  306530
3    70  ochotowo         2          2        1980        1  553641
4   136  mokotowo         3          2        1939        1  985348
(200, 6) (200,)
(200, 6) (200,)


In [62]:
# Convert categorical features to numbers (one ordinal column, for simplicity).
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

ordinal_encoder.fit(x_train[:, [1]])

x_train[:, [1]] = ordinal_encoder.transform(x_train[:, [1]])
x_test[:, [1]] = ordinal_encoder.transform(x_test[:, [1]])

# Convert ints to float.
x_train = x_train.astype(np.float64)
x_test = x_test.astype(np.float64)

# Preview the first three rows.
x_train[:3]

array([[ 104.,    1.,    2.,    2., 1940.,    1.],
       [  43.,    2.,    1.,    1., 1970.,    1.],
       [ 128.,    0.,    3.,    2., 1916.,    1.]])

## The loss, constant models

The range of prices is quite large.

In [63]:
print(np.min(y_train), np.max(y_train), np.mean(y_train))

102572 1102309 507919.49


When predicting them, relative errors are arguably more important than absolute errors.<br>
That is, mistaking 1M vs 2M is arguably similar to mistaking 11M vs 22M, more so than mistaking 11M vs 12M.

Therefore, let's assume that our predictions should minimize the so-called *mean squared logarithmic error*:
$$
MSLE(\bar{y}, \bar{p}) = \frac{1}{n} \sum_{i=1}^n (\log(1+y_i) - \log(1+p_i))^2,
$$
where $y_i$ is the ground truth (the target), and $p_i$ is our prediction.

Let's implement the loss function first.

In [64]:
def mse(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    return np.mean((ys - ps) * (ys - ps))

#
def rmse(ys: NDArray, ps: NDArray) -> np.float64:
    return np.sqrt(mse(ys, ps))

In [65]:
def msle(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    #################################################
    # TODO: Implement this function (without loops) #
    #################################################
    return np.mean((np.log(1 + ys) - np.log(1 + ps)) ** 2)


The simplest model is predicting the same constant for each instance. Test your implementation of msle against outputing the mean price.

In [66]:
###################################################
# TODO: Compute msle for outputing the mean price #
###################################################

min_price = np.min(y_train)
max_price = np.max(y_train)
mean_price = np.mean(y_train)
preds_test = np.full(y_test.shape, mean_price)
avg_price = np.mean(y_train)
print("test RMSE:", rmse(y_test, preds_test))
print("test MSLE:", msle(y_test, preds_test))


test RMSE: 293565.51772482495
test MSLE: 0.4284115392580848


To get a feel of whether this error is high or low, we can apply exp and sqrt to get some kind of mean (`exp of quadratic mean of log`) in interpretable units.<br>
(For context, observe that `exp of arithmetic mean of log` is the geometric mean).

In [67]:
np.exp(np.sqrt(msle(y_test, preds_test)))

np.float64(1.9242408798532662)

So we are roughly 1.92× off the actual y_test when we predict that constant value all the time, on average.

Recall that outputing the **mean** minimizes $MSE$. However, we're now dealing

---

with $MSLE$.

Find a constant that should result in the lowest $MSLE$.

In [68]:
#############################################
# TODO: Find this constant and compute msle #
#############################################


import numpy as np

c = np.expm1(np.mean(np.log1p(y_train)))
preds_test = np.full(y_test.shape, c)
print("test MSLE:", msle(y_test, preds_test))


test MSLE: 0.42757520337601673


**My note** :
 set

$$MSLE(c) = \frac{1}{n}\sum_i \big(\log(1+y_i) - \log(1+c)\big)^2$$

differentiate on c, and so on...



## Linear regression (standard)

Now, let's implement training of a standard linear regression model via gradient descent.

In [78]:
def train(
    x: NDArray, y: NDArray, alpha: float = 1e-7, n_iterations: int = 100000
) -> tuple[NDArray, np.float64]:
    """Linear regression (which optimizes MSE). Returns (weights, bias)."""

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)

    ### TODO ###

    weights = np.zeros(D)
    bias = np.float64(0.0)

    for i in range(n_iterations):

        preds = x @ weights + bias
        err = preds - y

        weights_grad = (2 * (x.T @ err)) / B
        bias_grad = (2 * err.sum()) / B

        weights -= alpha * weights_grad
        bias -= alpha * bias_grad

        return weights, bias


weights, bias = train(x_train, y_train)
preds_test = x_test @ weights + bias  # shapes (B, D) @ (D,) + ()
print("test MSLE:", msle(y_test, preds_test))
# Should be much better than the constant model.

test MSLE: 0.45180582829065513


## Linear regression (MSLE)

Note that the loss function that the algorithms optimizes (i.e $MSE$) differs from $MSLE$. We've already seen that this may result in a suboptimal solution.

How can you change the setting so that we optimze $MSLE$ instead?

Hint:
<sub><sup><sub><sup><sub><sup>
Be lazy. We don't want to change the algorithm.
Use the chain rule and previous computations to get formulas for the gradient.
</sup></sub></sup></sub></sup></sub>

**My note:**
The gradient had the structure:

$$\frac{\partial L}{\partial w} = X^\top \frac{\partial L}{\partial p}, \qquad \frac{\partial L}{\partial b} = \sum_i \frac{\partial L}{\partial p_i}$$

This holds for any loss, because $p = Xw + b$ is still the model. For MSE, $\frac{\partial L}{\partial p_i} = \frac{2}{B}(p_i - y_i)$. For MSLE you only need a new $\frac{\partial L}{\partial p_i}$:

$$L = \frac{1}{B}\sum_i \big(\log(1+p_i) - \log(1+y_i)\big)^2$$

$$\frac{\partial L}{\partial p_i} = \frac{2}{B},\big(\log(1+p_i) - \log(1+y_i)\big)\cdot\frac{1}{1+p_i}$$


In [76]:
def train_msle(
    x: NDArray, y: NDArray, alpha: float = 1e+4, n_iterations: int = 50000
) -> tuple[NDArray, NDArray]:
    """Gradient descent for MSLE."""

    #############################################
    # TODO: Optimize msle and compare the error #
    #############################################

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)

    weights = np.zeros(D)
    bias = np.expm1(np.mean(np.log1p(y))) # take the constant from the previous task

    log_y = np.log1p(y)
    for i in range(n_iterations):
        preds = x @ weights + bias
        preds_grad = (2 / B) * (np.log1p(preds) - log_y) / (1 + preds)  # dL/dp

        weights_grad = x.T @ preds_grad
        bias_grad = preds_grad.sum()

        weights -= alpha * weights_grad
        bias -= alpha * bias_grad


    return weights, bias


weights, bias = train_msle(x_train, y_train)
preds_test = x_test @ weights + bias
print("train MSLE:", msle(y_train, x_train @ weights + bias))
print("test MSLE: ", msle(y_test, preds_test))
# Train MSLE should be better. Test MSLE not necessarily.

train MSLE: 0.05396123439059387
test MSLE:  0.07994207226386915
